In [ ]:
!pip install kagglehub lexical-diversity vaderSentiment textstat tqdm

import os
import re
import ast
import zipfile
import importlib.metadata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import kagglehub
import sklearn
import textstat
from tqdm import tqdm
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from lexical_diversity import lex_div as ld

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

tqdm.pandas()


Downloads the Genius Song Lyrics dataset from Kaggle, cleans the lyric text, keeps
English-language songs, and restricts the data to the five genres analysed.

In [ ]:

genius_path = kagglehub.dataset_download(
    "carlosgdcj/genius-song-lyrics-with-language-information"
)

# Extract the lyrics
zip_path = os.path.join(genius_path, "song_lyrics.csv")
extract_folder = os.path.join(genius_path, "extracted")
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_folder)


genius_csv = os.path.join(extract_folder, "song_lyrics.csv")
df = pd.read_csv(genius_csv, nrows=3000000)
print(f"Loaded {len(df):,} rows from the Genius dataset")

In [ ]:
# Keep relevant columns
df = df[['title', 'artist', 'lyrics', 'language', 'tag', 'year']]
df = df.dropna(subset=['lyrics'])
df = df[df['language'] == 'en']

# Clean lyric text remove section tags, collapse whitespace
def clean_lyrics(text):
    text = re.sub(r"\[.*?\]", "", text, flags=re.DOTALL)
    text = text.replace("[", "").replace("]", "")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n+", "\n", text)
    return text.strip()
df['lyrics_clean'] = df['lyrics'].apply(clean_lyrics)
df = df[df['lyrics_clean'].str.strip() != ""]
df = df[df['lyrics_clean'].str.lower() != "nan"]

# Drop the 'misc' tag and keep the five genres analysed
df = df[df['tag'] != 'misc'].copy()
df = df[df['tag'].isin(['pop', 'rap', 'rock', 'country', 'rb'])].copy()
print(f"Genius songs after cleaning and genre filtering: {len(df):,}")

Downloads the Spotify audio-features dataset


In [ ]:

spotify_path = kagglehub.dataset_download("rodolfofigueroa/spotify-12m-songs")


spotify = pd.read_csv(
    os.path.join(spotify_path, "tracks_features.csv"), nrows=1200000
)

def get_main_artist(text):
    try:
        artists = ast.literal_eval(text)
        return artists[0] if len(artists) > 0 else ""
    except:
        return str(text)

spotify['artist_main'] = spotify['artists'].apply(get_main_artist)
print(f"Loaded {len(spotify):,} rows from the Spotify dataset")

## 3. Matching datasets

In [ ]:

def normalize_text(text):
    text = str(text).lower().strip()
    text = re.sub(r"[^a-z0-9]+", "", text)
    return text

spotify['artist_key'] = spotify['artist_main'].apply(normalize_text)
spotify['title_key']  = spotify['name'].apply(normalize_text)
df['artist_key'] = df['artist'].apply(normalize_text)
df['title_key']  = df['title'].apply(normalize_text)

merged = pd.merge(df, spotify, on=['title_key', 'artist_key'], how='inner')
print(f"Matched songs: {len(merged):,}")

# Keep the columns needed for the analysis
df = merged[['title', 'artist', 'lyrics_clean', 'tag',
             'tempo', 'duration_ms', 'time_signature', 'mode']].copy()
df = df.rename(columns={'tag': 'genre'})
df['duration_sec'] = df['duration_ms'] / 1000
df = df.drop(columns=['duration_ms'])

## 4. Lyric preprocessing 


In [ ]:

df['lyrics_no_newlines'] = df['lyrics_clean'].str.replace("\n", " ")

# Deduplicated lyrics, repeated lines removed
def remove_repetition(text):
    lines = str(text).split("\n")
    seen = set()
    unique_lines = []
    for line in lines:
        if line not in seen:
            unique_lines.append(line)
            seen.add(line)
    return " ".join(unique_lines)

df['lyrics_no_repetition'] = df['lyrics_clean'].apply(remove_repetition)

# Sample 100,000 songs for analysis (fixed 
df = df.sample(100000, random_state=42).copy()
print(f"Working sample: {len(df):,} songs")

## 5. Emotional valence

In [ ]:
analyzer = SentimentIntensityAnalyzer()

def get_sentiment(text):
    return analyzer.polarity_scores(str(text))['compound']


df['vader_original'] = df['lyrics_clean'].progress_apply(get_sentiment)


df['vader_no_repetition'] = df['lyrics_no_repetition'].progress_apply(get_sentiment)

## 6. Structural repetition and readability 

In [ ]:
#
def repetition_score(text):
    lines = [line.strip() for line in str(text).split("\n") if line.strip()]
    if len(lines) == 0:
        return 0
    unique_lines = set(lines)
    return 1 - (len(unique_lines) / len(lines))

df['repetition_score'] = df['lyrics_clean'].progress_apply(repetition_score)

def lyrics_to_flesch(text):
    lines = [line.strip() for line in str(text).split("\n") if line.strip()]
    unique_lines = list(dict.fromkeys(lines))
    as_sentences = ". ".join(unique_lines) + "."
    return textstat.flesch_reading_ease(as_sentences)

df['flesch_fixed'] = df['lyrics_clean'].progress_apply(lyrics_to_flesch)

## 7. Data cleaning for modelling



In [ ]:
df = df[df['flesch_fixed'] >= 0].copy()
df = df[df['tempo'] > 0].copy()
df = df[(df['duration_sec'] >= 30) & (df['duration_sec'] <= 600)].copy()
df = df[df['time_signature'] > 0].copy()

print(f"Songs remaining after cleaning: {len(df):,}")
print(df[['tempo', 'time_signature', 'duration_sec']].describe().round(2))

## 8. Lexical diversity MTLD

In [ ]:
def compute_mtld(text):
    words = str(text).lower().split()
    if len(words) < 10:
        return None
    try:
        return ld.mtld(words)
    except:
        return None

df['mtld'] = df['lyrics_no_newlines'].progress_apply(compute_mtld)
df = df[df['mtld'].notna()].copy()
df = df[df['mtld'] <= 200].copy()
print(f"Final sample: {len(df):,} songs")

In [ ]:
# Save the analysis-ready dataset
df.to_csv("dataset_final.csv", index=False)

## 9. statistics

In [ ]:
print("=== Sample Composition ===")
print(f"Total songs after filtering: {len(df):,}")
print("\nGenre distribution:")
print(df['genre'].value_counts())
print("\nGenre distribution (%):")
print((df['genre'].value_counts() / len(df) * 100).round(1))

In [ ]:
print("=== Feature Ranges ===")
print(df[['tempo', 'time_signature', 'duration_sec']].describe().round(2))

In [ ]:
print("=== Target Distributions ===")
print(df['vader_original'].describe().round(3))
print(df['repetition_score'].describe().round(3))
print(df['mtld'].describe().round(2))

print("\n=== Target Correlations ===")
print(df[['vader_original', 'repetition_score', 'mtld']].corr().round(3))

In [ ]:
print("=== Genre-level target means ===")
print(df.groupby('genre')[['vader_original', 'vader_no_repetition',
                           'repetition_score', 'mtld']].mean().round(3))

In [ ]:

print(df['flesch_fixed'].describe().round(2))
print(f"Flesch IQR: "
      f"{df['flesch_fixed'].quantile(0.75) - df['flesch_fixed'].quantile(0.25):.2f}")

## 10. Figure 1

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4), dpi=300)

panels = [
    {'col': 'vader_original',   'title': 'VADER (emotional valence)',
     'xlabel': 'VADER compound score', 'bins': np.linspace(-1, 1, 41),
     'color': '#2c6fbb'},
    {'col': 'repetition_score', 'title': 'Repetition Score (line-level redundancy)',
     'xlabel': 'Proportion of repeated lines', 'bins': np.linspace(0, 1, 31),
     'color': '#d97706'},
    {'col': 'mtld',             'title': 'MTLD (lexical diversity)',
     'xlabel': 'MTLD value', 'bins': np.linspace(0, 200, 41),
     'color': '#16a34a'},
]

for ax, p in zip(axes, panels):
    data = df[p['col']].dropna()
    ax.hist(data, bins=p['bins'], color=p['color'],
            edgecolor='white', linewidth=0.4)
    m, sd = data.mean(), data.std()
    ax.axvline(m, color='black', linestyle='--', linewidth=1, alpha=0.7)
    ax.text(0.97, 0.95, f'M = {m:.2f}\nSD = {sd:.2f}\nN = {len(data):,}',
            transform=ax.transAxes, ha='right', va='top', fontsize=9,
            bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
                      edgecolor='lightgray', alpha=0.9))
    ax.set_title(p['title'], fontsize=11)
    ax.set_xlabel(p['xlabel'], fontsize=10)
    ax.set_ylabel('Number of songs', fontsize=10)
    ax.set_axisbelow(True)
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('figure1_target_distributions.png',
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

## 11. Regression modelling


In [ ]:
FEATURES = ['tempo', 'time_signature', 'duration_sec']
TARGETS  = ['vader_original', 'repetition_score', 'mtld']

X = df[FEATURES]
results = {}

for target in TARGETS:
    y = df[target]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    models = {
        'Baseline (Mean)'  : DummyRegressor(strategy='mean'),
        'Linear Regression': Pipeline([('scaler', StandardScaler()),
                                       ('model', LinearRegression())]),
        'KNN'              : Pipeline([('scaler', StandardScaler()),
                                       ('model', KNeighborsRegressor(n_neighbors=10))]),
        'Random Forest'    : RandomForestRegressor(n_estimators=100,
                                                   random_state=42, n_jobs=-1),
    }
    results[target] = {}
    for name, model in models.items():
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        results[target][name] = {
            'MAE' : round(mean_absolute_error(y_test, preds), 4),
            'RMSE': round(root_mean_squared_error(y_test, preds), 4),
            'R2'  : round(r2_score(y_test, preds), 4),
        }
for target, model_res in results.items():
    print(f"\n{'='*55}")
    print(f"Target: {target}")
    print(f"{'='*55}")
    print(f"{'Model':<22} {'MAE':>8} {'RMSE':>8} {'R2':>8}")
    print(f"{'-'*55}")
    for name, metrics in model_res.items():
        print(f"{name:<22} {metrics['MAE']:>8} {metrics['RMSE']:>8} {metrics['R2']:>8}")

# Figure 2

In [ ]:
target_labels = {
    'vader_original':   'VADER\n(emotional valence)',
    'repetition_score': 'Repetition Score\n(line-level redundancy)',
    'mtld':             'MTLD\n(lexical diversity)',
}
models_order = ['Baseline (Mean)', 'Linear Regression', 'KNN', 'Random Forest']
colors = ['#7a7a7a', '#2c6fbb', '#d97706', '#16a34a']

targets = list(results.keys())
x = np.arange(len(targets))
bar_w = 0.20

fig, ax = plt.subplots(figsize=(8.5, 5.0), dpi=300)

for i, (m, c) in enumerate(zip(models_order, colors)):
    vals = [results[t][m]['R2'] for t in targets]
    offset = (i - (len(models_order) - 1) / 2) * bar_w
    legend_label = 'Baseline' if m == 'Baseline (Mean)' else m
    bars = ax.bar(x + offset, vals, bar_w, label=legend_label,
                  color=c, edgecolor='white', linewidth=0.5)
    for b, v in zip(bars, vals):
        y_off = 0.004 if v >= 0 else -0.004
        va = 'bottom' if v >= 0 else 'top'
        ax.text(b.get_x() + b.get_width()/2, v + y_off, f'{v:.3f}',
                ha='center', va=va, fontsize=8)

ax.axhline(0, color='black', linewidth=0.7)
ax.set_xticks(x)
ax.set_xticklabels([target_labels[t] for t in targets], fontsize=10)
ax.set_ylabel('R2 on held-out test set', fontsize=11)
ax.set_ylim(-0.12, 0.04)
ax.set_axisbelow(True)
ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(loc='lower left', frameon=False, fontsize=9, ncol=4,
          bbox_to_anchor=(0.0, -0.22))

plt.tight_layout()
plt.savefig('figure2_model_comparison.png',
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

## 13. Results

In [ ]:
rows = []
target_display = {
    'vader_original':   'VADER',
    'repetition_score': 'Repetition Score',
    'mtld':             'MTLD',
}

for target, model_res in results.items():
    for model, metrics in model_res.items():
        rows.append({
            'Target': target_display[target],
            'Model':  model,
            'MAE':    metrics['MAE'],
            'RMSE':   metrics['RMSE'],
            'R2':     metrics['R2'],
        })
table = pd.DataFrame(rows)
best_idx = table.groupby('Target')['R2'].idxmax()
table['_best'] = False
table.loc[best_idx, '_best'] = True

print("| Target | Model | MAE | RMSE | R2 |")
print("| --- | --- | --- | --- | --- |")
for _, r in table.iterrows():
    mae, rmse = f"{r['MAE']:.2f}", f"{r['RMSE']:.2f}"
    r2 = f"{r['R2']:.3f}"
    if r['_best']:
        r2 = f"**{r2}**"
    print(f"| {r['Target']} | {r['Model']} | {mae} | {rmse} | {r2} |")
table_out = table.drop(columns=['_best']).copy()
table_out['MAE']  = table_out['MAE'].round(2)
table_out['RMSE'] = table_out['RMSE'].round(2)
table_out['R2']   = table_out['R2'].round(3)
table_out.to_csv('table_4_1_results.csv', index=False)
print("\nTable saved to: table_4_1_results.csv")

## 14. VADER full-versus-deduplicated 


In [ ]:
print(f"Full:          {df['vader_original'].mean():.4f}")
print(f"Deduplicated:  {df['vader_no_repetition'].mean():.4f}")
print(f"Difference:    {(df['vader_original'] - df['vader_no_repetition']).mean():.4f}")
print(f"Mean abs diff: {(df['vader_original'] - df['vader_no_repetition']).abs().mean():.4f}")

## 15. Library version

In [ ]:
print(f"pandas:            {pd.__version__}")
print(f"numpy:             {np.__version__}")
print(f"scikit-learn:      {sklearn.__version__}")
print(f"vaderSentiment:    {importlib.metadata.version('vaderSentiment')}")
print(f"textstat:          {textstat.__version__}")
print(f"lexical-diversity: {importlib.metadata.version('lexical-diversity')}")
print(f"tqdm:              {importlib.metadata.version('tqdm')}")
print(f"kagglehub:         {importlib.metadata.version('kagglehub')}")